In [3]:
import os
import sys
import opendatasets as od 
import pandas as pd
from pyspark.sql import SparkSession

In [4]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

## Import data

In [3]:
od.download("https://www.kaggle.com/datasets/vipin20/transaction-data", data_dir="data") 

Skipping, found downloaded files in "data/transaction-data" (use force=True to force download)


In [5]:
spark = SparkSession.builder.appName("Sales Recommend").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/02 13:17:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
sales_df = spark.read.csv("data/transaction-data/transaction_data.csv", header=True, inferSchema=True)

In [7]:
sales_df.show(10)

+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|UserId|TransactionId|     TransactionTime|ItemCode|     ItemDescription|NumberOfItemsPurchased|CostPerItem|       Country|
+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|278166|      6355745|Sat Feb 02 12:50:...|  465549|FAMILY ALBUM WHIT...|                     6|      11.73|United Kingdom|
|337701|      6283376|Wed Dec 26 09:06:...|  482370|LONDON BUS COFFEE...|                     3|       3.52|United Kingdom|
|267099|      6385599|Fri Feb 15 09:45:...|  490728|SET 12 COLOUR PEN...|                    72|        0.9|        France|
|380478|      6044973|Fri Jun 22 07:14:...|  459186|UNION JACK FLAG L...|                     3|       1.73|United Kingdom|
|    -1|      6143225|Mon Sep 10 11:58:...| 1733592| WASHROOM METAL SIGN|                     3|        3.4|United Kingdom|
|285957|

24/12/02 13:18:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Clean data

In [6]:
sales_df.describe().show()

24/12/02 10:57:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+--------------------+-----------------+--------------------+----------------------+------------------+-----------+
|summary|            UserId|     TransactionId|     TransactionTime|         ItemCode|     ItemDescription|NumberOfItemsPurchased|       CostPerItem|    Country|
+-------+------------------+------------------+--------------------+-----------------+--------------------+----------------------+------------------+-----------+
|  count|           1083818|           1083818|             1083818|          1083818|             1080910|               1083818|           1083818|    1083818|
|   mean| 241016.2188245628| 6159416.640572495|                NULL|658268.6963124805|             20713.0|     28.65674864229972| 9.498797565644036|       NULL|
| stddev|142336.43126421853|147634.09388005832|                NULL|452631.4265032327|                 0.0|     654.2431717268586|2308.1385781550457|       NULL|
|    min|                -1|

24/12/02 10:57:59 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
from pyspark.sql import functions as F

In [9]:
sales_df_clean = sales_df.filter(
    (F.col('ItemCode') != -1) &
    (F.col('NumberOfItemsPurchased') > 0) &
    (F.col('CostPerItem') > 0)
)

In [10]:
df_user_minus_1 = sales_df_clean.filter(sales_df_clean['UserId'] == -1)

In [11]:
df_user_minus_1.groupBy('Country').count().orderBy(F.desc('count')).show()

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|260052|
|          EIRE|  1302|
|     Hong Kong|   558|
|   Unspecified|   404|
|   Switzerland|   250|
|        France|   132|
|        Israel|    94|
|      Portugal|    78|
|       Bahrain|     2|
+--------------+------+



In [12]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Group by 'Country' and 'ItemDescription' and sum the 'NumberOfItemsPurchased'
top_items_by_country_user_minus_1 = df_user_minus_1.groupBy('Country', 'ItemDescription').agg(
    F.sum('NumberOfItemsPurchased').alias('TotalItemsPurchased')
)

# For each country, order by 'TotalItemsPurchased' and get the top 10 items
window_spec = Window.partitionBy('Country').orderBy(F.desc('TotalItemsPurchased'))

# Add a row number to each row based on the country and order by total items purchased
top_items_by_country = top_items_by_country_user_minus_1.withColumn('rank', F.row_number().over(window_spec))

# Show the schema to verify the columns
top_items_by_country.printSchema()

# Filter to get only the top 10 items for each country
top_10_items_by_country = top_items_by_country.filter(F.col('rank') <= 10)

# Show the result
top_10_items_by_country.select('Country', 'ItemDescription', 'TotalItemsPurchased', 'rank') \
    .where(F.col('Country') == 'United Kingdom') \
    .orderBy('Country', 'rank') \
    .show(truncate=False)


root
 |-- Country: string (nullable = true)
 |-- ItemDescription: string (nullable = true)
 |-- TotalItemsPurchased: long (nullable = true)
 |-- rank: integer (nullable = false)

+--------------+-------------------------------+-------------------+----+
|Country       |ItemDescription                |TotalItemsPurchased|rank|
+--------------+-------------------------------+-------------------+----+
|United Kingdom|CHARLOTTE BAG SUKI DESIGN      |55002              |1   |
|United Kingdom|POPCORN HOLDER                 |34818              |2   |
|United Kingdom|RED RETROSPOT CHARLOTTE BAG    |29676              |3   |
|United Kingdom|WOODLAND CHARLOTTE BAG         |24246              |4   |
|United Kingdom|PAPER CHAIN KIT 50'S CHRISTMAS |22428              |5   |
|United Kingdom|RABBIT NIGHT LIGHT             |21222              |6   |
|United Kingdom|PARTY BUNTING                  |17886              |7   |
|United Kingdom|STRAWBERRY CHARLOTTE BAG       |17706              |8   |
|United

In [13]:
sales_df_clean = sales_df_clean.filter(sales_df_clean['UserId'] != -1)

### Remove outlier

In [14]:
def remove_outliers_iqr(df, column):
    # Calculate Q1, Q3, and IQR
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1, Q3 = quantiles
    IQR = Q3 - Q1

    # Define lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter out rows outside the bounds
    filtered_df = df.filter((F.col(column) >= lower_bound) & (F.col(column) <= upper_bound))
    return filtered_df

In [15]:
removed_outliers_df = remove_outliers_iqr(sales_df_clean, 'NumberOfItemsPurchased')
removed_outliers_df = remove_outliers_iqr(removed_outliers_df, 'CostPerItem')

In [16]:
removed_outliers_df.show()

+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|UserId|TransactionId|     TransactionTime|ItemCode|     ItemDescription|NumberOfItemsPurchased|CostPerItem|       Country|
+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|337701|      6283376|Wed Dec 26 09:06:...|  482370|LONDON BUS COFFEE...|                     3|       3.52|United Kingdom|
|267099|      6385599|Fri Feb 15 09:45:...|  490728|SET 12 COLOUR PEN...|                    72|        0.9|        France|
|380478|      6044973|Fri Jun 22 07:14:...|  459186|UNION JACK FLAG L...|                     3|       1.73|United Kingdom|
|285957|      6307136|Fri Jan 11 09:50:...| 1787247|CUT GLASS T-LIGHT...|                    12|       3.52|United Kingdom|
|345954|      6162981|Fri Sep 28 10:51:...|  471576|NATURAL SLATE CHA...|                     9|       6.84|United Kingdom|
|339822|

### Cap outliers

In [32]:
def cap_outliers_iqr(df, column):
    # Calculate Q1, Q3, and IQR
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1, Q3 = quantiles
    IQR = Q3 - Q1

    # Define lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap the outliers
    filtered_df = df.withColumn(column, 
                                F.when(F.col(column) < lower_bound, lower_bound)
                                .when(F.col(column) > upper_bound, upper_bound)
                                .otherwise(F.col(column)))
    return filtered_df

In [33]:
cap_outliers_df = cap_outliers_iqr(sales_df_clean, 'NumberOfItemsPurchased')
cap_outliers_df = cap_outliers_iqr(cap_outliers_df, 'CostPerItem')

In [35]:
cap_outliers_df.show()

+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|UserId|TransactionId|     TransactionTime|ItemCode|     ItemDescription|NumberOfItemsPurchased|CostPerItem|       Country|
+------+-------------+--------------------+--------+--------------------+----------------------+-----------+--------------+
|278166|      6355745|Sat Feb 02 12:50:...|  465549|FAMILY ALBUM WHIT...|                   6.0|      7.605|United Kingdom|
|337701|      6283376|Wed Dec 26 09:06:...|  482370|LONDON BUS COFFEE...|                   3.0|       3.52|United Kingdom|
|267099|      6385599|Fri Feb 15 09:45:...|  490728|SET 12 COLOUR PEN...|                  72.0|        0.9|        France|
|380478|      6044973|Fri Jun 22 07:14:...|  459186|UNION JACK FLAG L...|                   3.0|       1.73|United Kingdom|
|285957|      6307136|Fri Jan 11 09:50:...| 1787247|CUT GLASS T-LIGHT...|                  12.0|       3.52|United Kingdom|
|345954|

### Collabration filtering

In [17]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS 
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [18]:
als = ALS(
    userCol="UserId",
    itemCol="ItemCode",
    ratingCol="NumberOfItemsPurchased",
    coldStartStrategy="drop"  # Avoid NaN during evaluation
)

In [19]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="NumberOfItemsPurchased",
    predictionCol="prediction"
)

In [20]:
(training_df, testing_df) = removed_outliers_df.randomSplit([0.8, 0.2], seed=42)
als_model = als.fit(training_df)
predictions = als_model.transform(testing_df)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error before 3-fold cross-validation = {rmse}")

24/12/02 13:18:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/12/02 13:18:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
24/12/02 13:18:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Root-mean-square error before 3-fold cross-validation = 9.487042709607513


In [37]:
# Create the parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(als.rank, [4, 10, 50]) \
    .addGrid(als.regParam, [0.01, 0.001, 0.1]) \
    .build()

# Instantiate the cross-validator
crossval = CrossValidator(
    estimator=als,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3  # 3-fold cross-validation
)

# Perform cross-validation to find the best model
cv_model = crossval.fit(training_df)

best_model = cv_model.bestModel

In [38]:
# Display the best rank and regularization parameter
print(f"Best rank: {best_model.rank}")
print(f"Best regularization parameter: {best_model._java_obj.parent().getRegParam()}")

# Evaluate the best model on the testing set
predictions = best_model.transform(testing_df)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error of the best model = {rmse}")

Best rank: 50
Best regularization parameter: 0.1


Root-mean-square error of the best model = 9.170358090501033


#### With dataset removing outliers:

Best rank: 50

Best regularization parameter: 0.1
                                                                        
Root-mean-square error of the best model = 7.870934278720618

#### With dataset capping outliers:

Best rank: 50

Best regularization parameter: 0.1
                                                                                
Root-mean-square error of the best model = 9.170358090501033

### Try with different parameter

In [40]:
newParamGrid = ParamGridBuilder()\
    .addGrid(als.rank, [10, 20, 50, 100])\
    .addGrid(als.regParam, [0.01, 0.001, 0.1, 0.0001])\
    .addGrid(als.maxIter, [10, 20, 50])\
    .build()

crossval = CrossValidator(
    estimator=als,
    estimatorParamMaps=newParamGrid,
    evaluator=evaluator,
    numFolds=3
)

In [39]:
(training_df, testing_df) = removed_outliers_df.randomSplit([0.8, 0.2], seed=42)

In [41]:
cv_model = crossval.fit(training_df)
best_model = cv_model.bestModel

In [42]:
print(f"Best rank: {best_model.rank}")
print(f"Best regParam: {best_model._java_obj.parent().getRegParam()}")
print(f"Best maxIter: {best_model._java_obj.parent().getMaxIter()}")

Best rank: 100
Best regParam: 0.1
Best maxIter: 50


In [43]:
predictions = best_model.transform(testing_df)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error of the best model = {rmse}")

Root-mean-square error of the best model = 7.461871757490047


Best rank: 100

Best regParam: 0.1

Best maxIter: 50

rmse: 7.461871757490047

In [44]:
training_df.describe("NumberOfItemsPurchased").show()

+-------+----------------------+
|summary|NumberOfItemsPurchased|
+-------+----------------------+
|  count|                532029|
|   mean|    22.569461439132077|
| stddev|    20.365161149246628|
|    min|                     3|
|    max|                    81|
+-------+----------------------+



In [24]:
def get_recommendations_for_user(userId, sales_df, num_recs=10):
    
    user_df = sales_df.select("UserID").filter(f"UserID = {userId}").distinct()
    als = ALS(
        rank=100,
        maxIter=50,
        regParam=0.1,
        userCol="UserId",
        itemCol="ItemCode",
        ratingCol="NumberOfItemsPurchased",
        coldStartStrategy="drop"  # Avoid NaN during evaluation
    )
    als_model = als.fit(sales_df)

    user_recs = als_model.recommendForUserSubset(user_df, num_recs)

    recommendations = user_recs.select("recommendations").collect()[0]["recommendations"]
    print(recommendations)
    recommendation_list = []
    
    for idx, rec in enumerate(recommendations, start=1):
        predicted_score = rec["rating"]
        recommendation_list.append({
            "rank": idx,
            "ItemCode": rec["ItemCode"],
            "predicted_score": round(predicted_score, 6)
        })
    
    return recommendation_list

### Content-based fitering

In [52]:
from pyspark.ml.feature import Tokenizer
from pyspark.ml.feature import HashingTF, IDF
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import numpy as np

In [58]:
def build_vector_features(sales_df):
    item_df = sales_df.select("ItemCode", "ItemDescription").distinct()
    tokenizer = Tokenizer(inputCol="ItemDescription", outputCol="Words")
    df_tokenized = tokenizer.transform(item_df)
    
    hashing_tf = HashingTF(inputCol="Words", outputCol="RawFeatures", numFeatures=20)
    df_hashed = hashing_tf.transform(df_tokenized)

    idf = IDF(inputCol="RawFeatures", outputCol="Features")
    idf_model = idf.fit(df_hashed)
    df_tfidf = idf_model.transform(df_hashed)
    return df_tfidf

In [61]:
def cosine_similarity(vec1, vec2):
    vec1 = vec1.toArray()
    vec2 = vec2.toArray()
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return float(dot_product / (norm1 * norm2)) if norm1 > 0 and norm2 > 0 else 0.0

# UDF
similarity_udf = F.udf(lambda vec1: cosine_similarity(vec1, broadcast_item_vector.value), FloatType())

def recommend_similar_items(item_code, df_tfidf, top_n=10):
    
    input_vector = df_tfidf.filter(F.col("ItemCode") == item_code).select("ItemCode", "Features").first()
    df_tfidf = df_tfidf.filter(F.col("ItemCode") != item_code)
    global broadcast_item_vector
    broadcast_item_vector = spark.sparkContext.broadcast(input_vector["Features"])

    content_candidates = df_tfidf.withColumn(
        "cosine_similarity", similarity_udf(F.col("Features"))
    )
    top_similar_items = content_candidates.orderBy(F.col("cosine_similarity").desc()).limit(top_n)
    return top_similar_items

In [62]:
df_tfidf = build_vector_features(sales_df_clean)
recommendations = recommend_similar_items(465549, df_tfidf, top_n=10)

print(f"Top recommendations for ItemCode 465549:")
for rec in recommendations.collect():
    print(f"ItemCode: {rec['ItemCode']}, Similarity: {rec['cosine_similarity']:.4f}")


Top recommendations for ItemCode 465549:
ItemCode: 477393, Similarity: 0.9007
ItemCode: 744303, Similarity: 0.8670
ItemCode: 483126, Similarity: 0.8608
ItemCode: 1528947, Similarity: 0.8540
ItemCode: 487746, Similarity: 0.8522
ItemCode: 1786281, Similarity: 0.8522
ItemCode: 1893717, Similarity: 0.8171
ItemCode: 1893822, Similarity: 0.8166
ItemCode: 1893948, Similarity: 0.8162
ItemCode: 1894032, Similarity: 0.8141
